# F.R.I.D.A.Y. Voice Engine (XTTSv2) — Dual Voice Mode

**Instructions:**
1. Click `Runtime > Change runtime type` and select **T4 GPU**.
2. In the left sidebar (Files folder icon), upload your **two** reference voice MP3s:
   - `speaker_female.mp3` — The female voice you want F.R.I.D.A.Y. to clone
   - `speaker_male.mp3` — The male voice you want F.R.I.D.A.Y. to clone
3. Run all cells below. The script will install dependencies, download the model, and start a public URL tunnel.
4. Copy the `https://....loca.lt` URL printed at the bottom.
5. In F.R.I.D.A.Y.'s terminal, type: `/voice-engine https://YOUR_URL_HERE`

> Note: Google Colab may disconnect after a few hours of inactivity. Re-run this notebook when you start a new session.

In [ ]:
!pip install -q TTS fastapi uvicorn nest-asyncio
!npm install -g localtunnel

In [ ]:
import os
import torch
from TTS.api import TTS
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import subprocess
import threading
import time

print("Loading XTTSv2 Model... This will take a moment to download ~2GB.")
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("WARNING: Running on CPU. Voice generation will be extremely slow. Please enable T4 GPU in Runtime settings.")
else:
    print(f"SUCCESS: GPU detected ({torch.cuda.get_device_name(0)})! Real-time generation enabled.")

# Init TTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("Model loaded successfully!")

# Validate speaker files
SPEAKER_FILES = {
    "female": "speaker_female.mp3",
    "male": "speaker_male.mp3"
}

for label, fpath in SPEAKER_FILES.items():
    if os.path.exists(fpath):
        print(f"  ✓ {label} speaker loaded: {fpath}")
    else:
        print(f"  ✗ {label} speaker MISSING: {fpath} — Upload it to the Files panel!")

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    voice: str = "female"  # "male" or "female"

@app.post("/tts")
def generate_speech(req: TTSRequest):
    voice_key = req.voice.lower().strip()
    if voice_key not in SPEAKER_FILES:
        voice_key = "female"  # fallback

    speaker_path = SPEAKER_FILES[voice_key]
    if not os.path.exists(speaker_path):
        raise HTTPException(
            status_code=400,
            detail=f"{speaker_path} not found. Upload it to the Colab Files panel."
        )

    output_path = f"output_{voice_key}.wav"
    tts.tts_to_file(
        text=req.text,
        speaker_wav=speaker_path,
        language="en",
        file_path=output_path
    )

    return FileResponse(output_path, media_type="audio/wav")

@app.get("/")
def read_root():
    available = {k: os.path.exists(v) for k, v in SPEAKER_FILES.items()}
    return {"status": "F.R.I.D.A.Y. Voice Engine Online", "voices": available, "device": device}

# Start localtunnel in background
def start_localtunnel():
    process = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)
    for line in process.stdout:
        decoded = line.decode('utf-8').strip()
        if "your url is:" in decoded:
            url = decoded.replace('your url is: ', '')
            print("\n" + "="*60)
            print(f"\n🚀 COPY THIS URL FOR F.R.I.D.A.Y.:\n{url}\n")
            print("In F.R.I.D.A.Y.'s terminal type:")
            print(f"  /voice-engine {url}\n")
            print("="*60 + "\n")

threading.Thread(target=start_localtunnel, daemon=True).start()
time.sleep(3)

# Run FastAPI
nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)
